<a href="https://colab.research.google.com/github/jerperkins/zhuang-dialect-analysis/blob/main/CSC575ZhuangProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install Praat-parselmouth - only needed the first time.
!pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 60.7 MB/s eta 0:00:00


In [12]:
from google.colab import drive
import pandas as pd
import ipywidgets as widgets
import librosa
import librosa.display
import os
# Import praat-parselmouth to use Praat's pitch tracking algorithm
# Needs installing once via !pip install praat-parselmouth (using the above cell)
import parselmouth
import matplotlib.pyplot as plt
import numpy as np
# UI that allows filtering, displaying sounds nicely
from IPython.display import display, clear_output, Audio

# 1. Mount Drive (Force remount to see the new shortcut)
drive.mount('/content/drive', force_remount=True)

# 2. Define the Path
PROJECT_PATH = '/content/drive/MyDrive/Zhuang'
METADATA_PATH = os.path.join(PROJECT_PATH, 'FileInfoForRIPA.csv')

# 3. specific check
if os.path.exists(PROJECT_PATH):
    print(f"✅ Found the Zhuang folder at: {PROJECT_PATH}")

    if os.path.exists(METADATA_PATH):
        df = pd.read_csv(METADATA_PATH)
        print("✅ Metadata loaded!")
        display(df.head())
    else:
        print(f"❌ Folder found, but could not find 'FileInfoForRIPA.csv' inside it.")
        print("Check if the file is still uploading?")
else:
    print(f"❌ Still cannot find the folder at: {PROJECT_PATH}")
    print("Did you create the shortcut in 'My Drive'?")

# ***********************************************

AUDIO_PATH = os.path.join(PROJECT_PATH, 'Audio')

# Create UI Widgets
tone_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['ReclassifiedTone'].dropna().unique().tolist()),
    value='All',
    description='Tone:'
)

vowel_length_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['ReclassifiedVowelLength'].dropna().unique().tolist()),
    value='All',
    description='Vowel Length:'
)

# This allows us to exclude audio files that were excluded due to errors
exclude_dropdown = widgets.Dropdown(
    options=['All'] + sorted(df['Exclude'].dropna().unique().tolist()),
    value='All',
    description='Exclude:'
)

# User can choose number of files to display at once
n_input = widgets.BoundedIntText(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Num Files:',
    layout=widgets.Layout(width='150px')
)

# Spectrogram? y/n
spectrogram_dropdown = widgets.Dropdown(
    options=['y', 'n'],
    value='y',
    description='Show Spec:',
    layout=widgets.Layout(width='150px')
)

# Create the manual trigger button to fix bottleneck issue with filters
search_button = widgets.Button(description="Plot Audio", button_style='success')
output_box = widgets.Output()

# Plotting function using the button click as input
def plot_audio_files(b):
  with output_box:
    clear_output(wait=True)

    # Grab the current values from the dropdowns
    tone = tone_dropdown.value
    vowel_length = vowel_length_dropdown.value
    exclude_status = exclude_dropdown.value
    n = n_input.value
    show_spectrogram = spectrogram_dropdown.value

    # Filter based on selections
    filtered_df = df.copy()
    if tone != 'All':
      filtered_df = filtered_df[filtered_df['ReclassifiedTone'] == tone]
    if vowel_length != 'All':
      filtered_df = filtered_df[filtered_df['ReclassifiedVowelLength'] == vowel_length]
    if exclude_status != 'All':
      filtered_df = filtered_df[filtered_df['Exclude'] == exclude_status]

    display_df = filtered_df.head(n)
    print(f"Found {len(filtered_df)} matching files. Displaying the first {n}...")
    print("Please wait, generating acoustic data...\n")

    # Limit to the first n files
    for i, (index, row) in enumerate(display_df.iterrows(), 1):
      # Construct the file path
      file_name = row['Filename'] + '.wav'
      audio_path = os.path.join(AUDIO_PATH, file_name)

      if not os.path.exists(audio_path):
        print(f"Missing audio file: {file_name}")
        continue

      print(f"\n--- File: {file_name} | Tone: {row['ReclassifiedTone']} | Vowel Length: {row['ReclassifiedVowelLength']}")

      # Load audio
      y, sr = librosa.load(audio_path, sr = None)

      # Load with parselmouth for f0
      snd = parselmouth.Sound(audio_path)
      pitch = snd.to_pitch()
      pitch_values = pitch.selected_array['frequency']
      pitch_times = pitch.xs()

      # Generate spectrogram
      D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

      # Create a 1-panel or 2-panel figure dynamically
      if show_spectrogram == 'y':
        fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=True)
        ax_wave = ax[0]
        ax_spec = ax[1]
      else:
        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 3))
        ax_wave = ax
        ax_spec = None

      # Panel 1 - waveform
      librosa.display.waveshow(y, sr = sr, ax = ax_wave)
      ax_wave.set(title = 'Waveform & f0', ylabel = 'Amplitude')

      # Overlay pitch track on panel 1
      pitch_values[pitch_values == 0] = np.nan # Hide voiceless segments
      ax_pitch = ax_wave.twinx() # create another y-axis sharing the same x-axis

      # Plot f0 - red makes it stand out
      ax_pitch.plot(pitch_times, pitch_values, 'o', markersize = 2, color = 'red')
      ax_pitch.set_ylabel('Pitch ($f_0$)', color = 'red')
      ax_pitch.set_ylim(50, 500)
      ax_pitch.tick_params(axis = 'y', colors = 'red')

      # Panel 2 - spectrogram
      if ax_spec is not None:
        librosa.display.specshow(D, sr = sr, x_axis = 'time', y_axis = 'hz', ax = ax[1])
        ax_spec.set(title = 'Spectrogram', ylabel = 'Hz')

      plt.tight_layout()
      plt.show()

      # Add audio player
      display(Audio(data = y, rate = sr))

# Button to run filter (avoiding instant run issues with multiple filters)
search_button.on_click(plot_audio_files)

# Display the UI
ui_row_1 = widgets.HBox([tone_dropdown, vowel_length_dropdown, exclude_dropdown, n_input, spectrogram_dropdown])
display(widgets.VBox([ui_row_1, search_button]))
display(output_box)

Mounted at /content/drive
✅ Found the Zhuang folder at: /content/drive/MyDrive/Zhuang
✅ Metadata loaded!


,Filename,OriginalIPA,ActualTone,VowelLength,IPA,Coda,ReclassifiedTone,ReclassifiedVowelLength,VowelQuality,PhonemicIPA,ReclassifiedIPA,ReclassifiedCoda,Exclude,DictionaryMatch,Onset,OnsType
0,WZ03_0_193_1,fɯːt8,0,long,fɯːt8,obstruent,0,long,ɯə,fɯːət,fɯːt,obstruent,Keep,yes,f,voiceless fricative
1,WZ03_3_74_1,poːn3,3,long,poːn3,nasal,3,long,o,poːn,pɔːn,nasal,Keep,yes,p,voiceless stop
2,WZ03_5_131_1,puən5_or_tiŋ2,5,short,puən5,nasal,5,long,uə,puːən,puːən,nasal,Keep,yes,p,voiceless stop
3,WZ03_9_182_1,θɯk7_or_ɕaːt9,9,long,ɕaːt9,obstruent,9,long,NaN,NaN,ɕaːt,obstruent,Skipped,NaN,ɕ,voiceless fricative
4,WZ03_1_9_1,ɣon1,1,short,ɣon1,nasal,1,short,o,ɣon,hon,nasal,Keep,yes,h,voiceless fricative


Output()